# Frontier × cuOpt — GPU exact solve, certify, and solver-exact duals

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cafzal/frontier/blob/main/examples/cuopt_colab.ipynb)

Frontier's two exact backends are co-equal: **HiGHS (CPU)** runs anywhere — including the hosted beta — while **cuOpt (GPU)** needs NVIDIA hardware, which is why it lives in this Colab template rather than the hosted deploy.

This notebook runs the full exact-audit arc on the bundled **investment portfolio** example (30 ETFs, mean-variance QP — cuOpt's headline shape):

1. explore with the default NSGA search,
2. re-solve exactly on the GPU (`solver="cuopt"`),
3. **certify** the heuristic frontier against the exact overlay,
4. read the **solver-exact duals** (`explore sensitivity`).

In normal use an AI agent drives these same four tools over MCP; here we call the tool functions in-process so everything runs inside one Colab runtime.

**Before running:** `Runtime → Change runtime type → GPU` (a T4 is plenty).

In [ ]:
# GPU sanity check — you want to see an NVIDIA device here.
!nvidia-smi -L

## Setup

Clone + editable install (the bundled `examples/` resolve relative to the repo root), then cuOpt from NVIDIA's index. The cuOpt wheel is large — expect a few minutes.

In [ ]:
!git clone --depth 1 https://github.com/cafzal/frontier.git
%cd frontier
!pip install -q -e .
!pip install -q --extra-index-url=https://pypi.nvidia.com cuopt-cu12

## Load the example and confirm cuOpt is on the table

`model load` rebuilds a bundled problem by name — scores, covariance interaction matrix, scenarios and all. (Tool responses also carry `_skill_guidance` — the workflow guidance Frontier injects for agents; we just don't print it here.)

In [ ]:
from mcp_server import server as frontier

loaded = frontier.model(action="load", source="investment_portfolio")
PID = loaded["problem_id"]
loaded["status"]

In [ ]:
# `solvers` reports what THIS environment offers — on a GPU runtime with
# cuopt-cu12 installed, expect 'cuopt' in available and exact_fits_shape=True
# (proportional mean-variance QP is a supported exact shape).
frontier.solve(action="validate", problem_id=PID)["solvers"]

## 1 — Explore with the evolutionary default

NSGA is the workflow's center of gravity: fast, fits any shape, reveals the tradeoff structure. Seeded for reproducibility.

In [ ]:
nsga = frontier.solve(action="run", problem_id=PID, seed=42)
{k: nsga[k] for k in ("solver_used", "solutions_found", "frontier_complete")} | nsga["objective_ranges"]

## 2 — Exact GPU pass: `solver="cuopt"`

The exact run is stored as an **overlay** (`exact_run`) alongside the NSGA frontier — an auditor, not a replacement. Long solves run in the background; we ask for a handle immediately (`wait_seconds=0`) and poll, exactly as an agent would.

In [ ]:
import time

res = frontier.solve(action="run", problem_id=PID, solver="cuopt", seed=42, wait_seconds=0)
while res.get("status") == "running":
    print(f"  optimizing… {res['label']}  {res['elapsed_s']}s")
    time.sleep(3)
    res = frontier.solve(action="status", job_id=res["job_id"])
{k: res.get(k) for k in ("status", "solver_used", "solutions_found")}

## 3 — Certify: audit the heuristic against the exact overlay

No params — `certify` audits the current NSGA `run` against the `exact_run` overlay. Read it as: **dominance audit** (heuristic slack the exact front catches), **coverage** (hypervolume reclaimed), the **soundness invariant** (NSGA dominates no exact point), and **corner sharpening** (the convex min-risk corner is where a QP shines and heuristics wobble).

In [ ]:
cert = frontier.explore(action="certify", problem_id=PID)
print(cert["recommendation"])
{
    "dominated_fraction": cert["dominance_audit"]["nsga_dominated_fraction"],
    "coverage_reclaimed": (cert["coverage"] or {}).get("reclaimed_fraction"),
    "invariant_holds": cert["invariant"]["holds"],
    "headline_corner": cert["headline_corner"],
}

## 4 — Solver-exact duals: where to invest, what just missed

Continuous (QP/LP) exact runs carry true duals read straight from the solver. The payload names the `optimized_objective` every number is measured against; `where_to_invest` ranks binding-constraint shadow prices, `near_misses` ranks excluded assets by how close they are to entering.

In [ ]:
sens = frontier.explore(action="sensitivity", problem_id=PID)
print("optimized objective:", sens["optimized_objective"], "| source:", sens["source"], "| solver:", sens["frontier_source"]["solver"])
print("\nWhere to invest:")
for w in sens["where_to_invest"][:3]:
    print(f"  {w['lever']:<22} {w['shadow_price']:>10.4g}   {w['interpretation']}")
print("\nNear-misses:")
for n in sens["near_misses"][:3]:
    print(f"  {n['option']:<22} {n['reduced_cost']:>10.4g}   {n['interpretation']}")

## 5 — See it: heuristic field vs exact overlay

In [ ]:
import matplotlib.pyplot as plt

base = frontier.explore(action="tradeoffs", problem_id=PID)["viz_data"]
exact = frontier.explore(action="tradeoffs", problem_id=PID, source="exact")["viz_data"]
ax_x, ax_y = 0, 1  # first two objectives (return vs volatility)
names = [o["name"] for o in base["objectives"]]

plt.figure(figsize=(7, 5))
plt.scatter([p["values"][ax_x] for p in base["points"]],
            [p["values"][ax_y] for p in base["points"]],
            s=18, alpha=0.35, label=f"NSGA ({base['provenance']['kind']})")
plt.scatter([p["values"][ax_x] for p in exact["points"]],
            [p["values"][ax_y] for p in exact["points"]],
            s=34, marker="D", alpha=0.9, label=f"cuOpt ({exact['provenance']['kind']})")
plt.xlabel(names[ax_x]); plt.ylabel(names[ax_y])
plt.title("Heuristic field vs exact overlay — the min-risk corner is where exact earns its keep")
plt.legend(); plt.show()

## Notes & troubleshooting

- **Reproducibility:** Frontier runs cuOpt MILPs in deterministic mode automatically (`CUOPT_MIP_DETERMINISM_MODE`) — the seeded-reproducibility contract. The QP here is deterministic by nature.
- **`exact_fits_shape: false`?** The shape gate declines what the exact math can't represent (e.g. a *maximize* quadratic, or `min`/`max` aggregations) — with a redefine hint. The NSGA frontier remains the answer for those shapes.
- **Slow first solve:** the cuOpt wheel + first GPU kernel launch dominate; subsequent solves are fast.
- **Going deeper:** `frontier.get_skill("optimization_strategy", section="Exact Solvers — Depth")` is the agent-facing guidance on when an exact pass earns its cost and how to narrate it; the [architecture's Solver Backends section](https://github.com/cafzal/frontier/blob/main/architecture.md#solver-backends-pluggable) covers the engine design (one scalarization engine, two inner solves).
- On CPU-only machines the identical arc runs with `solver="highs"` (`pip install "frontier[highs]"`) — same certificate, same duals.